# NB01 — Setup + Register Custom Env + Smoke Test

This notebook ensures that the **custom `UnitreeG1DishWipe-v1` environment** is correctly
registered and functional.  We verify all dependencies, create the environment, discover its
specification (obs/act spaces, active joints, TCP, contact forces, dirt grid) and run a short
smoke test.

**Key change from original pipeline:** We now use a **real custom environment** with:
- A kitchen counter scene with sink (3D model)
- A physical plate **in the kitchen sink**
- The Unitree G1 **upper body** robot (25 DOF, fixed legs)
- Real contact detection between hand and plate
- Integrated VirtualDirtGrid for cleaning progress tracking

## Objective

1. Verify all dependencies are installed and importable.
2. Register and create the `UnitreeG1DishWipe-v1` environment in **joint-control** mode.
3. Extract observation/action space dimensions.
4. Discover active joints (25 DOF upper body).
5. Verify **TCP availability** (left_tcp, right_tcp).
6. Verify **real contact force** detection (multi-link: palm + 3 fingers ↔ plate).
7. Test dirt grid integration.
8. Run a 50-step smoke test with random actions.
9. Attempt to render one frame.
10. Log all results to MLflow (with CSV fallback).


## Environment

### Resource Requirements for NB01

| Resource | Minimum | Notes |
|----------|---------|-------|
| GPU | None | CPU-only smoke test |
| RAM | 4 GB | Env creation + 50 steps |
| Disk | 500 MB | Asset download |
| Time | < 2 min | Full notebook run |

In [1]:
# ── Print system & library versions ──
import sys, platform
print(f"Python  : {sys.version}")
print(f"Platform: {platform.platform()}")
print(f"Machine : {platform.machine()}")

Python  : 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.22631-SP0
Machine : AMD64


## Imports

We import all libraries needed throughout this notebook in one place.
The custom environment is registered when `src.envs` is imported.

In [2]:
# ── Core imports ──
import os, sys, json, csv, random, datetime, platform, traceback, subprocess
from pathlib import Path

import numpy as np
import torch
import gymnasium as gym

# ManiSkill
import mani_skill
import sapien

# Ensure project root is on sys.path so `src.envs` can be imported
# If running from notebooks/, go up one level to find src/
PROJECT_ROOT = Path(os.getcwd()).resolve()
if (PROJECT_ROOT / "src").is_dir():
    pass  # already at project root
elif (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError(f"Cannot find src/ from {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # normalize cwd for artifact paths

# Register custom env by importing the package
from src.envs.dishwipe_env import UnitreeG1DishWipeEnv  # triggers @register_env
from src.envs.dirt_grid import VirtualDirtGrid

# Versions
versions = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "torch": torch.__version__,
    "gymnasium": gym.__version__,
    "mani_skill": mani_skill.__version__,
    "sapien": sapien.__version__,
}
for k, v in versions.items():
    print(f"  {k:12s}: {v}")

# Confirm registration
assert "UnitreeG1DishWipe-v1" in gym.envs.registry, "Custom env not registered!"
print("\n✅ UnitreeG1DishWipe-v1 registered in gymnasium")


c:\Users\Siripon Sri\Desktop\My Project\robotic-sim\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Siripon Sri\Desktop\My Project\robotic-sim\.env\Lib\site-packages\sapien\__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
c:\Users\Siripon Sri\Desktop\My Project\robotic-sim\.env\Lib\site-packages\sapien\wrapper\pinocchio_model.py:299: UserWarning: pinnochio package is not installed, robotics functionalities will not be available
  warnings.warn(


  python      : 3.14.0
  numpy       : 2.4.2
  torch       : 2.10.0+cpu
  gymnasium   : 0.29.1
  mani_skill  : 3.0.0b22
  sapien      : 3.0.2

✅ UnitreeG1DishWipe-v1 registered in gymnasium


### MLflow Setup (with CSV fallback)

We initialise MLflow **once** here so every downstream cell can call `mlflow.log_*`
inside the same run.  If MLflow is unreachable we fall back to a local CSV file.

In [3]:
# ── MLflow initialisation (try/except — never crashes the notebook) ──
MLFLOW_ACTIVE = False
try:
    from dotenv import load_dotenv
    env_file = PROJECT_ROOT / ".env.local"
    if env_file.exists():
        load_dotenv(env_file)
        print(f"Loaded secrets from {env_file}")
    else:
        print(f"⚠️ {env_file} not found — MLflow may not authenticate")

    import mlflow
    tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
    if tracking_uri:
        mlflow.set_tracking_uri(tracking_uri)
        print(f"MLflow tracking URI: {tracking_uri}")
        MLFLOW_ACTIVE = True
    else:
        print("⚠️ MLFLOW_TRACKING_URI not set — using CSV fallback")
except Exception as e:
    print(f"⚠️ MLflow setup failed: {e}")

# CSV fallback writer
def csv_log(path, row_dict):
    """Append one row to a CSV file (creates header on first write)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists()
    with open(path, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row_dict.keys()))
        if write_header:
            w.writeheader()
        w.writerow(row_dict)

print(f"MLFLOW_ACTIVE = {MLFLOW_ACTIVE}")

Loaded secrets from C:\Users\Siripon Sri\Desktop\My Project\robotic-sim\.env.local
MLflow tracking URI: https://mlflow.cie.co.th
MLFLOW_ACTIVE = True


## Config

Single source-of-truth dictionary for every tuneable parameter in this notebook.
No magic numbers elsewhere — everything references `CFG`.

In [4]:
# ── Configuration (single source of truth) ──
CFG = dict(
    env_id           = "UnitreeG1DishWipe-v1",
    obs_mode         = "state",
    control_mode     = "pd_joint_delta_pos",
    render_mode      = "rgb_array",
    num_envs         = 1,
    seed             = 42,
    smoke_steps      = 50,
    artifact_dir     = "artifacts/NB01",
    mlflow_experiment = "dishwipe_unitree_g1",
    mlflow_run_name  = "NB01_setup_smoke_v2",
    csv_fallback     = "artifacts/NB01/nb01_log.csv",
)

for k, v in CFG.items():
    print(f"  {k:20s}: {v}")

  env_id              : UnitreeG1DishWipe-v1
  obs_mode            : state
  control_mode        : pd_joint_delta_pos
  render_mode         : rgb_array
  num_envs            : 1
  seed                : 42
  smoke_steps         : 50
  artifact_dir        : artifacts/NB01
  mlflow_experiment   : dishwipe_unitree_g1
  mlflow_run_name     : NB01_setup_smoke_v2
  csv_fallback        : artifacts/NB01/nb01_log.csv


## Reproducibility

Set seeds for **all** random sources so the smoke test produces the same sequence
of actions on every run.

In [5]:
# ── Set seeds ──
SEED = CFG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"All seeds set to {SEED}")

All seeds set to 42


## Implementation Steps

### Step 1 — Create Custom Environment & Discover Specs

We create the environment with `gym.make()`, verify the requested `control_mode` is
applied, and extract observation/action space metadata.

**Key differences from old pipeline (UnitreeG1Stand-v1):**
- Robot: `unitree_g1_simplified_upper_body` (25 DOF, fixed root)
- Scene: Kitchen counter + sink + plate (plate inside sink basin)
- Obs: ~168 dims (qpos + qvel + TCP + palm + plate + palm_to_plate + contact + cleaned_ratio + dirt_grid)
- Act: 25 dims (was 37)

In [6]:
# ── Step 1: Create env & discover specs ──
artifact_dir = Path(CFG["artifact_dir"])
artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Creating {CFG['env_id']}...")
env = gym.make(
    CFG["env_id"],
    num_envs=CFG["num_envs"],
    obs_mode=CFG["obs_mode"],
    control_mode=CFG["control_mode"],
    render_mode=CFG["render_mode"],
)
print("✅ Environment created!")

obs, info = env.reset(seed=SEED)

obs_space = env.observation_space
act_space = env.action_space

env_spec = {
    "env_id": CFG["env_id"],
    "obs_mode": CFG["obs_mode"],
    "control_mode": CFG["control_mode"],
    "obs_space_shape": list(obs_space.shape),
    "obs_dim": list(obs.shape),
    "act_space_shape": list(act_space.shape),
    "act_low": act_space.low[:3].tolist(),
    "act_high": act_space.high[:3].tolist(),
    "robot_uid": "unitree_g1_simplified_upper_body",
    "robot_dof": int(act_space.shape[0]),
    "scene": "kitchen_counter + sink + plate",
    "info_keys": list(info.keys()),
    "num_envs": CFG["num_envs"],
}

print("\n" + "=" * 60)
print(f"  Env ID         : {env_spec['env_id']}")
print(f"  Robot           : {env_spec['robot_uid']}")
print(f"  Robot DOF       : {env_spec['robot_dof']}")
print(f"  Obs space       : {env_spec['obs_space_shape']}")
print(f"  Obs dim (reset) : {env_spec['obs_dim']}")
print(f"  Act space       : {env_spec['act_space_shape']}")
print(f"  Act bounds      : [{env_spec['act_low'][0]:.1f}, {env_spec['act_high'][0]:.1f}]")
print(f"  Scene           : {env_spec['scene']}")
print(f"  Info keys       : {env_spec['info_keys']}")
print("=" * 60)

# Save env spec
spec_path = artifact_dir / "env_spec.json"
with open(spec_path, "w") as f:
    json.dump(env_spec, f, indent=2)
print(f"\n✅ Saved: {spec_path}")

2026-03-01 16:57:18,651 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


Creating UnitreeG1DishWipe-v1...
✅ Environment created!

  Env ID         : UnitreeG1DishWipe-v1
  Robot           : unitree_g1_simplified_upper_body
  Robot DOF       : 25
  Obs space       : [1, 168]
  Obs dim (reset) : [1, 168]
  Act space       : [25]
  Act bounds      : [-1.0, 1.0]
  Scene           : kitchen_counter + sink + plate
  Info keys       : ['elapsed_steps', 'success', 'fail', 'contact_force', 'delta_clean', 'cleaned_ratio', 'reconfigure']

✅ Saved: artifacts\NB01\env_spec.json


### Step 2 — Discover Active Joints

The `UnitreeG1UpperBody` agent has **25 active joints**: 1 torso + 11 upper body
(shoulder/elbow) + 14 finger joints (7 per hand).  Legs are **fixed** (not in action space).

In [7]:
# ── Step 2: Discover active joints ──
joint_info = {"source": None, "joint_names": [], "num_joints": 0}
agent = env.unwrapped.agent
robot = agent.robot

# Try active_joints_map
try:
    names = list(robot.active_joints_map.keys())
    joint_info["source"] = "robot.active_joints_map"
    joint_info["joint_names"] = names
    joint_info["num_joints"] = len(names)
except Exception:
    pass

# Fallback: try active_joints
if not joint_info["joint_names"]:
    try:
        joints = robot.get_active_joints()
        names = [j.name for j in joints]
        joint_info["source"] = "robot.get_active_joints()"
        joint_info["joint_names"] = names
        joint_info["num_joints"] = len(names)
    except Exception:
        pass

print(f"Source: {joint_info['source']}")
print(f"Number of active joints: {joint_info['num_joints']}")
print(f"\nJoint names:")
for i, name in enumerate(joint_info["joint_names"]):
    # Categorise
    if "shoulder" in name or "elbow" in name or "torso" in name:
        cat = "upper_body"
    elif any(x in name for x in ["zero", "one", "two", "three", "four", "five", "six"]):
        cat = "finger"
    else:
        cat = "other"
    print(f"  [{i:2d}] {name:40s}  ({cat})")

# Verify no leg joints
leg_joints = [n for n in joint_info["joint_names"] if "hip" in n or "knee" in n or "ankle" in n]
assert len(leg_joints) == 0, f"Unexpected leg joints in action space: {leg_joints}"
print(f"\n✅ No leg joints in action space (legs are fixed)")

# Save
joints_path = artifact_dir / "active_joints.json"
with open(joints_path, "w") as f:
    json.dump(joint_info, f, indent=2)
print(f"💾 Saved {joints_path}")

Source: robot.active_joints_map
Number of active joints: 25

Joint names:
  [ 0] torso_joint                               (upper_body)
  [ 1] left_shoulder_pitch_joint                 (upper_body)
  [ 2] right_shoulder_pitch_joint                (upper_body)
  [ 3] left_shoulder_roll_joint                  (upper_body)
  [ 4] right_shoulder_roll_joint                 (upper_body)
  [ 5] left_shoulder_yaw_joint                   (upper_body)
  [ 6] right_shoulder_yaw_joint                  (upper_body)
  [ 7] left_elbow_pitch_joint                    (upper_body)
  [ 8] right_elbow_pitch_joint                   (upper_body)
  [ 9] left_elbow_roll_joint                     (upper_body)
  [10] right_elbow_roll_joint                    (upper_body)
  [11] left_zero_joint                           (finger)
  [12] left_three_joint                          (finger)
  [13] left_five_joint                           (finger)
  [14] right_zero_joint                          (finger)
  [15] right

### Step 3 — Check TCP & Palm Links

Unlike the old `UnitreeG1Stand-v1` (which had **no TCP**), the `UnitreeG1UpperBody`
agent provides both `left_tcp` and `right_tcp` links.  The env v2 uses a
**force-weighted centroid** of contacting links (palm + 3 fingers) for grid mapping.
TCP is kept for reference; the reach reward targets the **palm** position.


In [8]:
# ── Step 3: Check TCP (end-effector) ──
tcp_info = {"tcp_available": False, "left_tcp": None, "right_tcp": None, "notes": ""}
obs, info = env.reset(seed=SEED)

try:
    left_tcp = agent.left_tcp
    right_tcp = agent.right_tcp
    tcp_info["tcp_available"] = True
    tcp_info["left_tcp"] = {
        "link_name": left_tcp.name,
        "position": left_tcp.pose.p[0].cpu().numpy().tolist(),
    }
    tcp_info["right_tcp"] = {
        "link_name": right_tcp.name,
        "position": right_tcp.pose.p[0].cpu().numpy().tolist(),
    }
    tcp_info["notes"] = "Both left and right TCP available"
    print("✅ TCP is AVAILABLE!")
    print(f"  Left TCP  link: {tcp_info['left_tcp']['link_name']}")
    print(f"  Left TCP  pos : {tcp_info['left_tcp']['position']}")
    print(f"  Right TCP link: {tcp_info['right_tcp']['link_name']}")
    print(f"  Right TCP pos : {tcp_info['right_tcp']['position']}")
except AttributeError as e:
    tcp_info["notes"] = f"TCP not available: {e}"
    print(f"⚠️ TCP not available: {e}")

# Also check palm links
try:
    left_palm = robot.links_map["left_palm_link"]
    right_palm = robot.links_map["right_palm_link"]
    tcp_info["left_palm_pos"] = left_palm.pose.p[0].cpu().numpy().tolist()
    tcp_info["right_palm_pos"] = right_palm.pose.p[0].cpu().numpy().tolist()
    print(f"  Left palm pos : {tcp_info['left_palm_pos']}")
    print(f"  Right palm pos: {tcp_info['right_palm_pos']}")
except Exception as e:
    print(f"  Palm links: {e}")

# Save
tcp_path = artifact_dir / "tcp_info.json"
with open(tcp_path, "w") as f:
    json.dump(tcp_info, f, indent=2, default=str)
print(f"\n💾 Saved {tcp_path}")

✅ TCP is AVAILABLE!
  Left TCP  link: left_tcp_link
  Left TCP  pos : [-0.026593660935759544, 0.12538133561611176, 0.8431978225708008]
  Right TCP link: right_tcp_link
  Right TCP pos : [-0.014477035962045193, -0.11532579362392426, 0.8393742442131042]
  Left palm pos : [-0.097383514046669, 0.1586763709783554, 0.8409086465835571]
  Right palm pos: [-0.0833025872707367, -0.15258222818374634, 0.8394069075584412]

💾 Saved artifacts\NB01\tcp_info.json


### Step 4 — Check Contact Force API (Multi-link ↔ Plate)

The custom environment uses **multi-link contact** — it queries
`scene.get_pairwise_contact_forces()` for **4 links** (left_palm_link +
left_two_link, left_four_link, left_six_link) against the plate, then
computes a force-weighted centroid for dirt-grid mapping.
Here we verify the API works for at least the palm link.


In [9]:
# ── Step 4: Contact force API discovery ──
contact_info = {"api_found": False, "method": None, "notes": []}
action = env.action_space.sample()
obs, reward, terminated, truncated, info = env.step(action)

# Method 1: scene.get_pairwise_contact_forces (primary method)
try:
    uw = env.unwrapped
    palm_link = uw.agent.robot.links_map["left_palm_link"]
    plate = uw.plate
    forces = uw.scene.get_pairwise_contact_forces(palm_link, plate)
    contact_info["api_found"] = True
    contact_info["method"] = "scene.get_pairwise_contact_forces(palm_link, plate)"
    contact_info["force_shape"] = list(forces.shape)
    contact_info["force_sample"] = forces[0].cpu().numpy().tolist()
    contact_info["notes"].append("Pairwise contact forces between palm and plate available")
    print(f"✅ Contact force API found!")
    print(f"   Method: {contact_info['method']}")
    print(f"   Shape : {forces.shape}")
    print(f"   Sample: {forces[0].cpu().numpy()}")
except Exception as e:
    contact_info["notes"].append(f"Pairwise contact forces failed: {e}")
    print(f"⚠️ Pairwise contact failed: {e}")

# Method 2: Check info dict for contact_force key
if "contact_force" in info:
    cf = info["contact_force"]
    contact_info["notes"].append(f"info['contact_force'] available: {cf}")
    print(f"   info['contact_force'] = {cf}")

# Method 3: link.get_net_contact_forces (fallback)
try:
    palm_forces = palm_link.get_net_contact_forces()
    contact_info["notes"].append(f"link.get_net_contact_forces shape: {list(palm_forces.shape)}")
    print(f"   link.get_net_contact_forces: {palm_forces[0].cpu().numpy()}")
except Exception as e:
    contact_info["notes"].append(f"get_net_contact_forces failed: {e}")

# Save
cf_path = artifact_dir / "contact_force_notes.json"
with open(cf_path, "w") as f:
    json.dump(contact_info, f, indent=2, default=str)
print(f"\n💾 Saved {cf_path}")

✅ Contact force API found!
   Method: scene.get_pairwise_contact_forces(palm_link, plate)
   Shape : torch.Size([1, 3])
   Sample: [0. 0. 0.]
   info['contact_force'] = tensor([0.])
   link.get_net_contact_forces: [0. 0. 0.]

💾 Saved artifacts\NB01\contact_force_notes.json


### Step 5 — Test Dirt Grid Integration

The custom environment integrates a `VirtualDirtGrid(10×10)` that tracks
which areas of the plate have been cleaned.  We verify:
- Grid initialises to all-dirty (0%)
- Grid is accessible from `env.unwrapped._dirt_grids[0]`
- `cleaned_ratio` appears in info dict
- Grid resets on `env.reset()`

In [10]:
# ── Step 5: Test dirt grid integration ──
uw = env.unwrapped
obs, info = env.reset(seed=SEED)

# Check dirt grid exists
dg = uw._dirt_grids[0]
print(f"Dirt grid: {dg}")
print(f"  Grid shape   : {dg.grid.shape}")
print(f"  Brush radius : {dg.brush_radius}")
print(f"  Cleaned ratio: {dg.get_cleaned_ratio():.2%}")
assert dg.get_cleaned_ratio() == 0.0, "Grid should start all dirty!"

# Verify cleaned_ratio in info
print(f"  info['cleaned_ratio'] = {info.get('cleaned_ratio', 'NOT FOUND')}")
assert "cleaned_ratio" in info, "cleaned_ratio missing from info!"

# Test manual mark_clean
delta = dg.mark_clean(5, 5)
print(f"\n  mark_clean(5,5) -> delta_clean = {delta}")
print(f"  Cleaned ratio after manual clean: {dg.get_cleaned_ratio():.2%}")

# Reset and verify
dg.reset()
assert dg.get_cleaned_ratio() == 0.0, "Grid should be all dirty after reset!"
print(f"  After reset: {dg.get_cleaned_ratio():.2%}")

# Test coordinate mapping using palm position (env v2 uses palm for reach reward)
plate_pos = uw.plate.pose.p[0].cpu().numpy()
palm_pos = uw.agent.robot.links_map["left_palm_link"].pose.p[0].cpu().numpy()
from src.envs.dishwipe_env import PLATE_HALF_SIZE
plate_half = np.array(PLATE_HALF_SIZE[:2])
u, v = dg.world_to_uv(palm_pos, plate_pos, plate_half)
cell = dg.uv_to_cell(u, v)
print(f"\n  Plate center      : {plate_pos}")
print(f"  Left palm position: {palm_pos}")
print(f"  Palm -> (u, v)    : ({u:.3f}, {v:.3f})")
print(f"  Palm -> cell      : {cell}")

print("\n✅ Dirt grid integration verified")


Dirt grid: VirtualDirtGrid(H=10, W=10, brush_radius=1, cleaned=0.0%)
  Grid shape   : (10, 10)
  Brush radius : 1
  Cleaned ratio: 0.00%
  info['cleaned_ratio'] = tensor([0.])

  mark_clean(5,5) -> delta_clean = 9
  Cleaned ratio after manual clean: 9.00%
  After reset: 0.00%

  Plate center      : [0.09322052 0.21052144 0.58      ]
  Left palm position: [-0.09738351  0.15867637  0.84090865]
  Palm -> (u, v)    : (0.000, 0.241)
  Palm -> cell      : (2, 0)

✅ Dirt grid integration verified


### Step 6 — Smoke Test (50 random steps)

Reset the environment and step 50 times with random actions.
We collect basic statistics to verify the env runs without errors.

In [11]:
# ── Step 6: Smoke test — 50 random steps ──
obs, info = env.reset(seed=SEED)
print(f"Initial obs shape: {obs.shape}")

N = CFG["smoke_steps"]
rewards = []
contact_forces = []
cleaned_ratios = []
n_terminated = 0
n_contacts = 0

for step_i in range(N):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    
    r = float(reward) if isinstance(reward, (int, float)) else float(reward.item()) if hasattr(reward, 'item') else float(reward)
    rewards.append(r)
    
    cf = info.get("contact_force", torch.tensor([0.0]))
    cf_val = float(cf.item()) if hasattr(cf, 'item') else float(cf[0]) if hasattr(cf, '__len__') else float(cf)
    contact_forces.append(cf_val)
    if cf_val > 0.5:
        n_contacts += 1
    
    cr = info.get("cleaned_ratio", torch.tensor([0.0]))
    cr_val = float(cr.item()) if hasattr(cr, 'item') else float(cr[0]) if hasattr(cr, '__len__') else float(cr)
    cleaned_ratios.append(cr_val)
    
    if bool(terminated) if not hasattr(terminated, '__len__') else bool(terminated.any()):
        n_terminated += 1
        obs, info = env.reset(seed=SEED + step_i)

smoke_results = {
    "smoke_passed": True,
    "steps_completed": N,
    "mean_reward": float(np.mean(rewards)),
    "std_reward": float(np.std(rewards)),
    "min_reward": float(np.min(rewards)),
    "max_reward": float(np.max(rewards)),
    "cumulative_reward": float(np.sum(rewards)),
    "n_terminated": n_terminated,
    "n_contacts": n_contacts,
    "max_contact_force": float(np.max(contact_forces)),
    "final_cleaned_ratio": cleaned_ratios[-1] if cleaned_ratios else 0.0,
    "obs_shape": list(obs.shape),
}

print("\n" + "=" * 60)
print(f"  SMOKE TEST: {'✅ PASSED' if smoke_results['smoke_passed'] else '❌ FAILED'}")
print(f"  Steps completed  : {smoke_results['steps_completed']}")
print(f"  Mean reward      : {smoke_results['mean_reward']:.6f}")
print(f"  Std reward       : {smoke_results['std_reward']:.6f}")
print(f"  Cumulative reward: {smoke_results['cumulative_reward']:.4f}")
print(f"  Episodes ended   : {smoke_results['n_terminated']}")
print(f"  Contact events   : {smoke_results['n_contacts']} / {N}")
print(f"  Max contact force: {smoke_results['max_contact_force']:.2f} N")
print(f"  Final cleaned    : {smoke_results['final_cleaned_ratio']:.2%}")
print("=" * 60)

# Save
smoke_path = artifact_dir / "smoke_summary.json"
with open(smoke_path, "w") as f:
    json.dump(smoke_results, f, indent=2)
print(f"\n💾 Saved {smoke_path}")

Initial obs shape: torch.Size([1, 168])

  SMOKE TEST: ✅ PASSED
  Steps completed  : 50
  Mean reward      : -0.005927
  Std reward       : 0.001271
  Cumulative reward: -0.2963
  Episodes ended   : 0
  Contact events   : 0 / 50
  Max contact force: 0.00 N
  Final cleaned    : 0.00%

💾 Saved artifacts\NB01\smoke_summary.json


### Step 7 — Render Test (try/except with graceful fallback)

Rendering may fail with Vulkan `ErrorOutOfPoolMemory` on machines without a
proper GPU.  We wrap `env.render()` in a `try/except`.  Training does **not**
depend on rendering.

In [12]:
# ── Step 7: Render test with graceful fallback ──
render_success = False
video_saved = False

try:
    obs, info = env.reset(seed=SEED)
    frame = env.render()
    if frame is not None:
        if hasattr(frame, 'cpu'):
            frame = frame.cpu().numpy()
        if isinstance(frame, np.ndarray) and frame.size > 0:
            render_success = True
            print(f"✅ Render success! Frame shape: {frame.shape}, dtype: {frame.dtype}")
            
            # Try saving as image
            try:
                from PIL import Image
                if frame.ndim == 4:  # batched
                    frame = frame[0]
                if frame.dtype != np.uint8:
                    frame = np.clip(frame * 255, 0, 255).astype(np.uint8)
                img = Image.fromarray(frame)
                img_path = artifact_dir / "render_test.png"
                img.save(img_path)
                print(f"💾 Saved render frame to {img_path}")
            except Exception as e:
                print(f"⚠️ Could not save render image: {e}")
        else:
            print("⚠️ Render returned empty frame")
    else:
        print("⚠️ Render returned None")
except Exception as e:
    print(f"⚠️ Render failed (expected on CPU-only machines): {e}")
    print("   Training does NOT depend on rendering.")

print(f"\nrender_success = {render_success}")

⚠️ Render failed (expected on CPU-only machines): vk::Device::allocateDescriptorSetsUnique: ErrorOutOfPoolMemory
   Training does NOT depend on rendering.

render_success = False


### Step 8 — Log to MLflow (with CSV fallback)

We log everything needed for reproducibility and debugging:

| Category | Items |
|----------|-------|
| Tags | project, nb, env_id, control_mode, platform |
| Params | seed, obs_dim, act_dim, control_mode, robot_uid, robot_dof, versions |
| Metrics | smoke_passed, mean_reward, n_contacts, final_cleaned_ratio, render_ok |
| Artifacts | env_spec.json, active_joints.json, tcp_info.json, contact_force_notes.json, smoke_summary.json |

In [13]:
# ── Step 8: Log to MLflow (or CSV fallback) ──

# ---- 8a. Auto-generate requirements.txt via pip freeze ----
try:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        capture_output=True, text=True, timeout=30,
    )
    req_path = artifact_dir / "requirements.txt"
    req_path.write_text(result.stdout)
    print(f"💾 Saved {req_path} ({len(result.stdout.splitlines())} packages)")
except Exception as e:
    print(f"⚠️ pip freeze failed: {e}")

# ---- 8b. Log to MLflow ----
if MLFLOW_ACTIVE:
    try:
        mlflow.set_experiment(CFG["mlflow_experiment"])
        with mlflow.start_run(run_name=CFG["mlflow_run_name"]) as run:
            # Tags
            mlflow.set_tags({
                "project": "dishwipe_g1",
                "nb": "NB01",
                "env_id": CFG["env_id"],
                "control_mode": CFG["control_mode"],
                "platform": platform.platform(),
                "robot_uid": "unitree_g1_simplified_upper_body",
                "env_version": "v2_custom_env",
            })

            # Params
            mlflow.log_params({
                "seed": SEED,
                "obs_dim": str(env_spec["obs_dim"]),
                "act_dim": str(env_spec["act_space_shape"]),
                "control_mode": CFG["control_mode"],
                "robot_uid": "unitree_g1_simplified_upper_body",
                "robot_dof": env_spec["robot_dof"],
                "smoke_steps": CFG["smoke_steps"],
                "grid_h": 10,
                "grid_w": 10,
                "brush_radius": 1,
            })
            for k, v in versions.items():
                mlflow.log_param(f"ver_{k}", v)

            # Metrics
            mlflow.log_metrics({
                "smoke_passed": 1.0 if smoke_results["smoke_passed"] else 0.0,
                "smoke_steps_completed": float(smoke_results["steps_completed"]),
                "mean_reward_smoke": smoke_results["mean_reward"],
                "n_terminated": float(smoke_results["n_terminated"]),
                "n_contacts": float(smoke_results["n_contacts"]),
                "max_contact_force": smoke_results["max_contact_force"],
                "final_cleaned_ratio": smoke_results["final_cleaned_ratio"],
                "render_ok": 1.0 if render_success else 0.0,
                "tcp_available": 1.0 if tcp_info["tcp_available"] else 0.0,
            })

            # Artifacts
            for p in artifact_dir.iterdir():
                if p.is_file():
                    mlflow.log_artifact(str(p))
                    print(f"  📤 {p.name}")

            print(f"\n✅ MLflow run ID: {run.info.run_id}")
            print(f"   Experiment: {CFG['mlflow_experiment']}")
    except Exception as e:
        print(f"⚠️ MLflow logging failed: {e}")
        traceback.print_exc()
        MLFLOW_ACTIVE = False

# ---- 8c. CSV fallback ----
if not MLFLOW_ACTIVE:
    print("Using CSV fallback...")
    csv_row = {
        "timestamp": datetime.datetime.now().isoformat(),
        "env_id": CFG["env_id"],
        "robot_uid": "unitree_g1_simplified_upper_body",
        "robot_dof": env_spec["robot_dof"],
        "obs_dim": str(env_spec["obs_dim"]),
        "act_dim": str(env_spec["act_space_shape"]),
        **smoke_results,
        "render_ok": render_success,
        "tcp_available": tcp_info["tcp_available"],
    }
    csv_log(CFG["csv_fallback"], csv_row)
    print(f"💾 Saved CSV: {CFG['csv_fallback']}")

💾 Saved artifacts\NB01\requirements.txt (147 packages)
  📤 active_joints.json
  📤 contact_force_notes.json
  📤 env_spec.json
  📤 requirements.txt
  📤 smoke_summary.json
  📤 tcp_info.json

✅ MLflow run ID: 2a937a70d2dc4d1baf8ae524bd63cc3e
   Experiment: dishwipe_unitree_g1
🏃 View run NB01_setup_smoke_v2 at: https://mlflow.cie.co.th/#/experiments/7/runs/2a937a70d2dc4d1baf8ae524bd63cc3e
🧪 View experiment at: https://mlflow.cie.co.th/#/experiments/7


## Results

Summary of everything discovered and tested in this notebook.

In [14]:
# ── Results summary ──
print("\n" + "═" * 60)
print("NB01 — RESULTS SUMMARY")
print("═" * 60)

print(f"\n📋 Environment")
print(f"   ID            : {CFG['env_id']}")
print(f"   Robot          : unitree_g1_simplified_upper_body (25 DOF)")
print(f"   Scene          : Kitchen counter + sink + plate")
print(f"   Obs space      : {env_spec['obs_space_shape']}")
print(f"   Act space      : {env_spec['act_space_shape']}")
print(f"   Control mode   : {CFG['control_mode']}")

print(f"\n🦾 Robot")
print(f"   Active joints  : {joint_info['num_joints']}")
print(f"   TCP available  : {tcp_info['tcp_available']}")
print(f"   Contact API    : {contact_info['api_found']}")
print(f"   Fixed legs     : Yes (no balance needed)")

print(f"\n🧪 Smoke Test")
print(f"   Passed         : {smoke_results['smoke_passed']}")
print(f"   Steps          : {smoke_results['steps_completed']}")
print(f"   Mean reward    : {smoke_results['mean_reward']:.6f}")
print(f"   Episodes ended : {smoke_results['n_terminated']}")
print(f"   Contact events : {smoke_results['n_contacts']}")
print(f"   Cleaned ratio  : {smoke_results['final_cleaned_ratio']:.2%}")

print(f"\n🖼️ Render")
print(f"   Render OK      : {render_success}")

print(f"\n📊 MLflow")
print(f"   Active         : {MLFLOW_ACTIVE}")

print("\n" + "═" * 60)


════════════════════════════════════════════════════════════
NB01 — RESULTS SUMMARY
════════════════════════════════════════════════════════════

📋 Environment
   ID            : UnitreeG1DishWipe-v1
   Robot          : unitree_g1_simplified_upper_body (25 DOF)
   Scene          : Kitchen counter + sink + plate
   Obs space      : [1, 168]
   Act space      : [25]
   Control mode   : pd_joint_delta_pos

🦾 Robot
   Active joints  : 25
   TCP available  : True
   Contact API    : True
   Fixed legs     : Yes (no balance needed)

🧪 Smoke Test
   Passed         : True
   Steps          : 50
   Mean reward    : -0.005927
   Episodes ended : 0
   Contact events : 0
   Cleaned ratio  : 0.00%

🖼️ Render
   Render OK      : False

📊 MLflow
   Active         : True

════════════════════════════════════════════════════════════


## Artifacts

All artifacts are saved under `artifacts/NB01/`.

| File | Description |
|------|-------------|
| `env_spec.json` | Env ID, obs/act dims, robot, scene |
| `active_joints.json` | 25 active joint names and source |
| `tcp_info.json` | TCP and palm link positions |
| `contact_force_notes.json` | Contact API verification |
| `smoke_summary.json` | Smoke test results |
| `requirements.txt` | pip freeze snapshot |
| `render_test.png` | Render frame (if available) |

In [15]:
# ── List saved artifacts ──
print(f"Artifacts in {artifact_dir.resolve()}:\n")
for p in sorted(artifact_dir.iterdir()):
    if p.is_file():
        size = p.stat().st_size
        print(f"  {p.name:35s}  {size:>8,} bytes")

Artifacts in C:\Users\Siripon Sri\Desktop\My Project\robotic-sim\artifacts\NB01:

  active_joints.json                        773 bytes
  contact_force_notes.json                  378 bytes
  env_spec.json                             629 bytes
  requirements.txt                        2,761 bytes
  smoke_summary.json                        404 bytes
  tcp_info.json                             625 bytes


## Notes / Troubleshooting

### Key Differences from Original Pipeline

| Aspect | Old (v1) | New (v2) |
|--------|----------|----------|
| Env | `UnitreeG1Stand-v1` | `UnitreeG1DishWipe-v1` |
| Robot | `unitree_g1_simplified_legs` (37 DOF) | `unitree_g1_simplified_upper_body` (25 DOF) |
| Scene | Ground plane only | Kitchen counter + sink + plate |
| TCP | Not available | ✅ left_tcp, right_tcp |
| Contact | Self/ground forces only | ✅ Multi-link (palm + 3 fingers) ↔ plate |
| Dirt grid | External virtual overlay | Integrated in env (100D in obs) |
| Legs | Free (must balance) | Fixed (no balance needed) |
| Obs dims | ~60 | ~168 (qpos + qvel + TCP + palm + plate + palm_to_plate + contact + cleaned_ratio + dirt_grid) |
| Reward | Simple contact reward | Staged: reach + contact + clean + sweep + penalties |

### Known Issues

- **Render**: Vulkan `ErrorOutOfPoolMemory` on CPU-only machines — rendering is optional, all code uses `try/except`.
- **GPU**: Training notebooks (NB06–NB09) require CUDA GPU for reasonable speed; target is RunPod RTX 4090+.

In [16]:
# ── Cleanup ──
env.close()
print("✅ Environment closed. NB01 finished successfully.")

✅ Environment closed. NB01 finished successfully.


## References

1. **ManiSkill Documentation** — Environment registration, control modes, SAPIEN integration: https://maniskill.readthedocs.io/
2. **Unitree G1 Specs** — 23 DOF EDU Standard, 5 joints/arm, 6/leg, 1 waist
3. **Custom Env Pattern** — Based on `UnitreeG1PlaceAppleInBowl-v1` source code in `mani_skill/envs/tasks/humanoid/humanoid_pick_place.py`
4. **VirtualDirtGrid** — Dirt tracking module at `src/envs/dirt_grid.py`
5. **DishWipe Env** — Custom environment at `src/envs/dishwipe_env.py`